# Getting Experimental Metadata from DANDI for textures
It can be helpful to view general information about the experimental sessions that produced your data. Since typically each NWB File represents one session, a dandiset's files can be examined to get an overview of each of the sessions. This can vary, depending on who produced the NWB file. In this notebook, NWB Files within one of the Allen Institute's datasets are opened and some basic information is used to make a table of the experimental sessions and their properties.

To clearly identify the sessions for the texture project we need to pull additional metadata entries

### Environment Setup
⚠️**Note: If running on a new environment, run this cell once and then restart the kernel**⚠️

In [1]:
import warnings
warnings.filterwarnings('ignore')

try:
    from databook_utils.dandi_utils import dandi_stream_open
except:
    !git clone https://github.com/AllenInstitute/openscope_databook.git
    %cd openscope_databook
    %pip install -e .

In [1]:
import os

import h5py
import pandas as pd
import remfile

from dandi import dandiapi
from pynwb import NWBHDF5IO

%matplotlib inline

### Getting Dandiset Metadata
To view other data, change `dandiset_id` to be the id of the dandiset you're interested in. If the dandiset is embargoed, set `dandi_api_key` to your DANDI API key. 

In [2]:
dandiset_id = "001461"
dandi_api_key = os.environ["DANDI_API_KEY"]

In [3]:
my_dandiset = dandiapi.DandiAPIClient(token=dandi_api_key).get_dandiset(dandiset_id)
print(f"Got dandiset {my_dandiset}")

Got dandiset DANDI:001461/draft


### Get NWB Info
Below are two definitions of thefunction `get_nwb_info`. These function are tailored to our NWB Files; Our *Ophys* and our *Ecephys* datasets respectively. It retrieves a series of important metadata values from the NWB file object. It is likely that the code for accessing the fields of interest to you will be slightly different for your files. This can easily be altered to extract any other information from an NWB file you want as long as you're familiar with the internal layout of your files. However, make sure to change the `columns` field in the pandas dataframe below to properly reflect any changes to the function. `nwb_type` can be set below to 'ephys' or 'ophys' depending on the type of the NWBs of interest.

In [4]:
nwb_type = "ophys" # or, "ephys"

In [7]:
# get experimental information from within ophys file
# getattr is used because not all nwb files have all properties. If not handled like this, errors will arise
def get_ophys_nwb_info(nwb):
        session_time = getattr(nwb, "session_start_time", None)

        metadata_obj = getattr(nwb, "lab_meta_data", {})
        metadata = metadata_obj.get("metadata", None)
        session_id = getattr(metadata, "ophys_session_id", None)
        experiment_id = getattr(metadata, "ophys_experiment_id", None)

        fov_height = getattr(metadata, "field_of_view_height", None)
        fov_width = getattr(metadata, "field_of_view_width", None)
        imaging_depth = getattr(metadata, "imaging_depth", None)
        group = getattr(metadata, "imaging_plane_group", None)
        group_count = getattr(metadata, "imaging_plane_group_count", None)
        container_id = getattr(metadata, "experiment_container_id", None)
        
        subject = getattr(nwb, "subject", None)
        specimen_name = getattr(subject, "subject_id", None)
        age = getattr(subject, "age", None)
        sex = getattr(subject, "sex", None)
        genotype = getattr(subject, "genotype", None)
        
        try: n_rois = nwb.processing["ophys"]["dff"].roi_response_series["traces"].data.shape[1]
        except: n_rois = None
        try: location = list(nwb.imaging_planes.values())[0].location
        except: location = None
        
        intervals = getattr(nwb, "intervals", {})
        if "behavior_presentations" in intervals:
            if intervals["behavior_presentations"][0].image_set[0] == "OpenScope-Texture_stage-A2.5_240404":
                stim_types = "active_single_exemplar"
            elif intervals["behavior_presentations"][0].image_set[0] == "OpenScope-Texture_ShiftedImages_TriangleDistribution_8_13_stage-B2.5_241016":
                stim_types = "active_shifting_exemplars"
            else:
                stim_types = "active_unknown"
                print(f"Unknown session type on: {nwb.session_id}") 
        elif "fam-0-smp-0.tiff_presentations" in intervals:
            stim_types = "passive"  
        else:
            stim_types = "unknown"
            print(f"Unknown session type on: {nwb.session_id}")
        
        stim_tables = [intervals[table_name] for table_name in intervals]
        # gets highest value among final "stop times" of all stim tables in intervals
        session_end = max([table.stop_time[-1] for table in stim_tables if len(table) > 1])

        return [session_time, specimen_name, session_id, experiment_id, container_id, group, group_count, imaging_depth, location, fov_height, fov_width, specimen_name, sex, age, genotype, stim_types, n_rois, session_end]

In [7]:
# get experimental information from within ecephys file
# getattr is used because not all nwb files have all properties. If not handled like this, errors will arise
def get_ephys_nwb_info(nwb):
        session_time = getattr(nwb, "session_start_time", None)

        subject = getattr(nwb, "subject", None)
        specimen_name = getattr(subject, "specimen_name", None)
        age = getattr(subject, "age_in_days", None)
        sex = getattr(subject, "sex", None)
        genotype = getattr(subject, "genotype", None)

        probes = set(getattr(nwb, "devices", {}).keys())
        units = getattr(nwb, "units", [])
        n_units = len(units) if hasattr(units, '__len__') else 0        
        
        intervals = getattr(nwb, "intervals", {})
        stim_types = set(intervals.keys())
        stim_tables = [intervals[table_name] for table_name in intervals]
        # gets highest value among final "stop times" of all stim tables in intervals
        session_end = max([table.stop_time[-1] for table in stim_tables if len(table) > 1])

        return [session_time, specimen_name, sex, age, genotype, probes, stim_types, n_units, session_end]

### Getting Table
Here, each relevant file in the dandiset is streamed and opened remotely to get the information of interest using the function `get_nwb_info`, defined above, and then it is added to a table of sessions and their metadata. Since some files are for specific probes rather than entire sessions, they are skipped.

In [8]:
nwb_table = []
# skip files that aren't main session files
files = [asset for asset in my_dandiset.get_assets() if "probe" not in asset.path]
# swap this with line above for one of our ophys dandisets
# files = [asset for asset in my_dandiset.get_assets() if "raw" not in asset.path]
n_files = len(files)
print(f"{n_files} files retrieved")

extract_nwb_info = get_ephys_nwb_info if nwb_type == "ephys" else get_ophys_nwb_info

for i, file in enumerate(files):
    print(f"Examining file {i+1}/{n_files}: {file.identifier}")    
    # get basic file metadata
    row = [file.identifier, file.size, file.path]
    
    base_url = file.client.session.head(file.base_download_url)
    file_url = base_url.headers["Location"]

    # open and read nwb file with streaming
    rem_file = remfile.File(file_url)
    h5py_file = h5py.File(rem_file, "r")
    io = NWBHDF5IO(file=h5py_file, mode="r", load_namespaces=True)
    nwb = io.read()

    # extract experimental info from within file
    row += extract_nwb_info(nwb)
    nwb_table.append(row)
    del nwb

    # don't run full loop if running in test environment
    if os.environ.get("TESTING", False):
        break


144 files retrieved
Examining file 1/144: 3a99f387-2af0-4ead-9af2-b3fb6319e966
Examining file 2/144: 7c7f7ac5-3252-42c2-8172-db7d8cea3a4b
Examining file 3/144: 6d082069-8960-4bf2-9bec-2c0e0d6c9d5f
Examining file 4/144: 004043f9-321e-445e-b695-85e04002f2f1
Examining file 5/144: 9db56b96-0a4a-4bb3-be7e-2fced9efd393
Examining file 6/144: cdfcd9d0-2de5-46b4-805f-a3f91dfee8b7
Examining file 7/144: 8f2de772-089e-4c2f-bdad-87600da96bb4
Examining file 8/144: 4068057d-9d2d-4f58-a105-7b1e5285504e
Examining file 9/144: c809ff72-2074-4c6e-bf91-a447261b81de
Examining file 10/144: e8a5b4fd-d88a-4b39-80c3-48399374b77e
Examining file 11/144: 1eed7d79-ffc5-4139-8736-48159a72dfc8
Examining file 12/144: ec8ec2f3-c09f-4e38-9bde-10137752c33e
Examining file 13/144: c5d4942c-7f9e-4287-b36a-4ee420e375b7
Examining file 14/144: 6d8ee037-671b-4d9a-82dd-18a5cb80f931
Examining file 15/144: e0523d3c-46bc-45a8-b1d2-8475ec2d5eb8
Examining file 16/144: 5b52d1c0-df79-4a91-a068-cbfaec589f6b
Examining file 17/144: 970a63

Examining file 137/144: 50abc53a-e239-42fd-9dc7-16394d31c579
Examining file 138/144: 4e84bcf0-e0c4-4d18-92f8-d77db4593403
Examining file 139/144: ad6aeb66-6253-4881-940e-06e62c25ebf9
Examining file 140/144: fa152afc-97f0-4903-a0c1-36540101ab37
Examining file 141/144: 3120ec6f-b3d7-42ef-904b-e90deccf8025
Examining file 142/144: 74567b97-79f3-4b57-956c-501b48d9b3f3
Examining file 143/144: e8ac0c33-4057-4f58-9642-bfb47d91993b
Examining file 144/144: 29007b2e-57aa-4f01-a673-5369c2df09a3


In [9]:
# convert table to pandas dataframe
if nwb_type == "ephys":
    dandiset_files = pd.DataFrame(nwb_table, columns=("identifier", "size", "path", "session_time", "sub_name", "sub_sex", "sub_age", "sub_genotype", "probes", "stim types", "#_units", "session_length"))
else:
    dandiset_files = pd.DataFrame(nwb_table, columns=("identifier", "size", "path", "session_time", "sub_name", "session_id", "experiment_id", "container_id", "group", "group_count", "imaging_depth", "location", "fov_height", "fov_width", "specimen_name", "sub_sex", "sub_age", "sub_genotype", "stim_types", "#_rois", "session_end"))
dandiset_files

,identifier,size,path,session_time,sub_name,session_id,experiment_id,container_id,group,group_count,...,location,fov_height,fov_width,specimen_name,sub_sex,sub_age,sub_genotype,stim_types,#_rois,session_end
0,3a99f387-2af0-4ead-9af2-b3fb6319e966,1729071576,sub-770196/sub-770196_ses-multiplane-ophys-770...,2025-03-19 14:16:39.177500-07:00,770196,None,None,None,None,None,...,Structure: VISal Depth: 161,None,None,770196,F,P167D,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai93(TITL-G...,active_single_exemplar,None,3917.756320
1,7c7f7ac5-3252-42c2-8172-db7d8cea3a4b,980469748,sub-770962/sub-770962_ses-multiplane-ophys-770...,2025-03-08 11:22:47.102736-08:00,770962,None,None,None,None,None,...,Structure: VISal Depth: 161,None,None,770962,M,P152D,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai93(TITL-G...,active_shifting_exemplars,None,3920.138760
2,6d082069-8960-4bf2-9bec-2c0e0d6c9d5f,1545544760,sub-766757/sub-766757_ses-multiplane-ophys-766...,2025-02-19 14:31:09.098968-08:00,766757,None,None,None,None,None,...,Structure: VISal Depth: 148,None,None,766757,M,P156D,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai93(TITL-G...,passive,None,3502.008335
3,004043f9-321e-445e-b695-85e04002f2f1,1535386552,sub-766757/sub-766757_ses-multiplane-ophys-766...,2025-02-21 11:52:07.454331-08:00,766757,None,None,None,None,None,...,Structure: VISal Depth: 144,None,None,766757,M,P158D,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai93(TITL-G...,passive,None,3495.192990
4,9db56b96-0a4a-4bb3-be7e-2fced9efd393,1671603180,sub-766757/sub-766757_ses-multiplane-ophys-766...,2025-02-17 09:51:19.800644-08:00,766757,None,None,None,None,None,...,Structure: VISal Depth: 140,None,None,766757,M,P154D,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai93(TITL-G...,active_shifting_exemplars,None,3918.313840
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
139,fa152afc-97f0-4903-a0c1-36540101ab37,1598986452,sub-770196/sub-770196_ses-multiplane-ophys-770...,2025-03-21 14:14:24.910581-07:00,770196,None,None,None,None,None,...,Structure: VISal Depth: 158,None,None,770196,F,P169D,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai93(TITL-G...,active_single_exemplar,None,3920.496550
140,3120ec6f-b3d7-42ef-904b-e90deccf8025,1960065116,sub-766757/sub-766757_ses-multiplane-ophys-766...,2025-02-05 14:14:02.646070-08:00,766757,None,None,None,None,None,...,Structure: VISal Depth: 144,None,None,766757,M,P142D,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai93(TITL-G...,active_single_exemplar,None,3916.145170
141,74567b97-79f3-4b57-956c-501b48d9b3f3,1344031012,sub-762551/sub-762551_ses-multiplane-ophys-762...,2025-01-27 10:07:52.846287-08:00,762551,None,None,None,None,None,...,Structure: VISal Depth: 142,None,None,762551,F,P155D,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai93(TITL-G...,active_shifting_exemplars,None,3919.738360
142,e8ac0c33-4057-4f58-9642-bfb47d91993b,1393133956,sub-762551/sub-762551_ses-multiplane-ophys-762...,2025-01-23 15:07:09.374731-08:00,762551,None,None,None,None,None,...,Structure: VISal Depth: 140,None,None,762551,F,P151D,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai93(TITL-G...,active_shifting_exemplars,None,3928.410900


In [10]:
if nwb_type == "ephys":
    m_count = len(dandiset_files["sub_sex"][dandiset_files["sub_sex"] == "M"])
    f_count = len(dandiset_files["sub_sex"][dandiset_files["sub_sex"] == "F"])
    sst_count = len(dandiset_files[dandiset_files["sub_genotype"].str.count("Sst") >= 1])
    pval_count = len(dandiset_files[dandiset_files["sub_genotype"].str.count("Pval") >= 1])
    wt_count = len(dandiset_files[dandiset_files["sub_genotype"].str.count("wt/wt") >= 1])
    print("Dandiset Overview:")
    print(len(dandiset_files), "dandiset_files")
    print(len(set(dandiset_files["sub_name"])), "subjects", m_count, "males,", f_count, "females")
    print(sst_count, "sst,", pval_count, "pval,", wt_count, "wt")
else:
    n_sessions = len(dandiset_files["session_id"].value_counts())
    subjects_info = dandiset_files.groupby(["sub_name", "sub_sex"]).size().reset_index().to_dict()
    m_count = len([sex for sex in subjects_info["sub_sex"].values() if sex == "M"])
    f_count = len([sex for sex in subjects_info["sub_sex"].values() if sex == "F"])
    print("Dandiset Overview:")
    print(len(dandiset_files), "dandiset_files")
    print(len(subjects_info["sub_name"]), "subjects", m_count, "males", f_count,"females")

Dandiset Overview:
144 dandiset_files
15 subjects 9 males 6 females


In [12]:
# output all session metadata to local CSV file
dandiset_files.to_csv("../../data/texture_sessions.csv")

### Selecting Files
**Pandas** syntax can be used to filter the table above and select individual sessions.

In [12]:
selected_files = dandiset_files[dandiset_files["size"] <= 2_500_000_000]
# selected_files = dandiset_files[dandiset_files["sub_sex"] == "F"]
# selected_files = dandiset_files[dandiset_files["#_units"] > 2900]
selected_files

,identifier,size,path,session_time,sub_name,sub_sex,sub_age,sub_genotype,probes,stim types,#_units,session_length
4,0073a783-5b41-42a8-882a-2960554d4e43,2108745405,sub-631570/sub-631570_ses-1194857009_ogen.nwb,2022-07-28 00:00:00-07:00,631570,F,92.0,Pvalb-IRES-Cre/wt;Ai32(RCL-ChR2(H134R)_EYFP)/wt,"{probeC, probeB, probeE, probeA, probeD, probe...","{spontaneous_presentations, ICwcfg1_presentati...",1789,7278.92195
7,f1a26076-0a3e-43c8-b138-5320bca7a23f,2469141920,sub-619296/sub-619296_ses-1187930705_ogen.nwb,2022-06-29 00:00:00-07:00,619296,M,154.0,Sst-IRES-Cre/wt;Ai32(RCL-ChR2(H134R)_EYFP)/wt,"{probeC, probeB, probeE, probeA, probeD, probe...","{spontaneous_presentations, ICwcfg1_presentati...",1918,7278.13709
10,e61edcf3-26e9-44e1-9017-3ee558197da5,2453451408,sub-619293/sub-619293_ses-1184980079_ogen.nwb,2022-06-16 00:00:00-07:00,619293,M,141.0,Sst-IRES-Cre/wt;Ai32(RCL-ChR2(H134R)_EYFP)/wt,"{probeC, probeB, probeE, probeA, probeD, probe...","{spontaneous_presentations, ICwcfg1_presentati...",2136,7454.53444


### Downloading Selected Files
To download the files, we use the same method that is explained in [Downloading an NWB File](./download_nwb.ipynb). This can be used with the paths from the selected sessions above to just download the files of interest.  Set `download_loc` to be the relative path of where the files should be downloaded. Note that if the files are large, this can take a long time.

In [13]:
download_loc = "."

In [14]:
selected_paths = set(selected_files.path)
selected_paths

{'sub-619293/sub-619293_ses-1184980079_ogen.nwb',
 'sub-619296/sub-619296_ses-1187930705_ogen.nwb',
 'sub-631570/sub-631570_ses-1194857009_ogen.nwb'}

In [15]:
for dandi_filepath in selected_paths:
    filename = dandi_filepath.split("/")[-1]
    file = my_dandiset.get_asset_by_path(dandi_filepath)
    file.download(f"{download_loc}/{filename}")
    print(f"Downloaded file to {download_loc}/{filename}")

Downloaded file to ./sub-631570_ses-1194857009_ogen.nwb
Downloaded file to ./sub-619293_ses-1184980079_ogen.nwb
Downloaded file to ./sub-631570_ses-1194857009_ogen.nwb
Downloaded file to ./sub-619296_ses-1187930705_ogen.nwb
